In [2]:
# ==========================================================
# ACTIVIDAD 8. Consulta completa a la SEC
# Descarga el historial de informes de cada empresa candidata
# y guarda un resumen. Si se interrumpe, al volver a ejecutar
# continúa donde se quedó.
# ==========================================================
import time
from pathlib import Path

import pandas as pd
import requests
from google.colab import drive

CORREO = "tu_correo@ejemplo.com"   # <-- escribe aquí tu correo de contacto (lo exige la SEC)
assert "@" in CORREO and "ejemplo" not in CORREO, "Escribe tu correo en la variable CORREO"
PAUSA = 0.2                                   # 5 consultas por segundo como máximo
CLASES_A_CONSULTAR = ["accion_ordinaria", "sin_clasificar", "revisar_trust"]

# Informe anual -> régimen de información del emisor
REGIMEN = {
    "10-K": "nacional", "10-K/A": "nacional", "10-KT": "nacional", "10-KT/A": "nacional",
    "20-F": "extranjero", "20-F/A": "extranjero", "40-F": "extranjero", "40-F/A": "extranjero",
    "N-CSR": "fondo", "N-CSRS": "fondo",
}

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")
CARPETA_SEC = CARPETA / "sec_edgar"
AVANCE = CARPETA_SEC / "submissions_resumen.csv"     # se guarda cada 200 consultas

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# ----------------------------------------------------------
# 1. Empresas (CIK) que hay que consultar
# ----------------------------------------------------------
emp = pd.read_csv(CARPETA / "emparejamiento_sec.csv", dtype=str, keep_default_na=False)
a_consultar = emp[emp["clase"].isin(CLASES_A_CONSULTAR) & (emp["cik"] != "")]
ciks = sorted(a_consultar["cik"].unique())

filas = []
if AVANCE.exists():                                   # reanudar una ejecución anterior
    previas = pd.read_csv(AVANCE, dtype=str, keep_default_na=False)
    filas = previas[previas["error"] == ""].to_dict("records")   # los errores se reintentan
hechos = {fila["cik"] for fila in filas}
pendientes = [cik for cik in ciks if cik not in hechos]

sin_cik = emp[emp["clase"].isin(CLASES_A_CONSULTAR) & (emp["cik"] == "")]
print("Tickers sin CIK en la SEC (no se pueden consultar):", sorted(sin_cik["ticker"]))
print("Tickers a cubrir:", len(a_consultar), "| CIK únicos:", len(ciks))
print("Ya consultados:", len(hechos), "| Pendientes:", len(pendientes),
      f"| Tiempo estimado: {len(pendientes) * 0.4 / 60:.0f}-{len(pendientes) * 0.8 / 60:.0f} min")

# ----------------------------------------------------------
# 2. Funciones de consulta y resumen
# ----------------------------------------------------------
sesion = requests.Session()
sesion.headers.update({"User-Agent": f"TFM EBIS {CORREO}", "Accept-Encoding": "gzip, deflate"})

def consultar(cik):
    """Descarga el historial de una empresa. Reintenta si la SEC pide esperar."""
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    for intento in range(1, 5):
        try:
            r = sesion.get(url, timeout=60)
        except requests.RequestException:
            time.sleep(5 * intento)
            continue
        if r.status_code == 200:
            return r.json(), ""
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(10 * intento)                 # espera cada vez más
            continue
        return None, f"HTTP {r.status_code}"
    return None, "reintentos agotados"

def resumir(cik, datos):
    """Extrae los campos que necesitamos y el último informe anual."""
    recientes = pd.DataFrame(datos["filings"]["recent"])
    columnas = [c for c in ["form", "filingDate", "fileNumber"] if c in recientes.columns]
    recientes = recientes[columnas].sort_values("filingDate", ascending=False)
    anuales = recientes[recientes["form"].isin(REGIMEN)]
    ultimo = anuales.iloc[0] if len(anuales) else None
    return {
        "cik": cik,
        "nombre_sec": datos.get("name") or "",
        "tipo_entidad": datos.get("entityType") or "",
        "sic": datos.get("sic") or "",
        "sector_sic": datos.get("sicDescription") or "",
        "constitucion": datos.get("stateOfIncorporation") or "",
        "ultimo_anual": "" if ultimo is None else ultimo["form"],
        "fecha_anual": "" if ultimo is None else ultimo["filingDate"],
        "expediente_anual": "" if ultimo is None else ultimo.get("fileNumber", ""),
        "regimen": "sin_informe_anual" if ultimo is None else REGIMEN[ultimo["form"]],
        "error": "",
    }

# ----------------------------------------------------------
# 3. Consultar todas las empresas pendientes
# ----------------------------------------------------------
inicio = time.time()
for numero, cik in enumerate(pendientes, start=1):
    datos, error = consultar(cik)
    filas.append(resumir(cik, datos) if datos else {"cik": cik, "error": error})
    time.sleep(PAUSA)
    if numero % 200 == 0 or numero == len(pendientes):
        pd.DataFrame(filas).to_csv(AVANCE, index=False)
        print(f"{numero}/{len(pendientes)} consultadas | {(time.time() - inicio) / 60:.1f} min")

res = pd.DataFrame(filas).fillna("")
res.loc[res["error"] != "", "regimen"] = "error_consulta"

# ----------------------------------------------------------
# 4. Resultados
# ----------------------------------------------------------
print("\n=== RESULTADO DE LAS CONSULTAS ===")
print("Empresas consultadas:", len(res), "| con error:", (res["error"] != "").sum())
print(res.loc[res["error"] != "", ["cik", "error"]].head(20).to_string(index=False))

print("\nRégimen según el último informe anual:")
print(res["regimen"].value_counts())
print("\nÚltimo informe anual:")
print(res["ultimo_anual"].value_counts())
print("\nTipo de entidad frente a régimen:")
print(pd.crosstab(res["tipo_entidad"], res["regimen"]))

tabla = a_consultar[["ticker", "clase", "cik", "nombre_valor"]].merge(res, on="cik", how="left")
print("\n=== CLASE PRELIMINAR FRENTE A RÉGIMEN (por ticker) ===")
print(pd.crosstab(tabla["clase"], tabla["regimen"], margins=True, margins_name="Total"))

print("\nSin informe anual (máx. 30):")
print(tabla.loc[tabla["regimen"] == "sin_informe_anual",
                ["ticker", "nombre_sec", "tipo_entidad", "sic"]].head(30).to_string(index=False))

extranjeras = tabla[(tabla["clase"] == "accion_ordinaria") & (tabla["regimen"] == "extranjero")]
print(f"\nAcciones 'ordinarias' que son emisores extranjeros: {len(extranjeras)} (15 al azar)")
print(extranjeras.sample(min(15, len(extranjeras)), random_state=42)
      [["ticker", "nombre_valor", "ultimo_anual"]].to_string(index=False))

# Señales oficiales para BDC y fideicomisos de regalías
bdc = tabla[tabla["expediente_anual"].str.startswith("814-")]
print(f"\nBDC (expediente 814-): {len(bdc)} (máx. 30)")
print(bdc[["ticker", "nombre_valor", "expediente_anual"]].head(30).to_string(index=False))

regalias = tabla[tabla["sic"].isin(["6792", "6795"])]
print(f"\nSIC 6792/6795 (regalías petroleras y mineras): {len(regalias)}")
print(regalias[["ticker", "nombre_valor", "sic"]].to_string(index=False))

print("\nTickers con SIC vacío:", (tabla["sic"] == "").sum())
print("Sectores SIC distintos entre los nacionales:",
      tabla.loc[tabla["regimen"] == "nacional", "sic"].nunique())

# ----------------------------------------------------------
# 5. Guardar
# ----------------------------------------------------------
res.to_csv(AVANCE, index=False)
tabla.to_csv(CARPETA / "clasificacion_sec.csv", index=False)
print("\nGuardados: sec_edgar/submissions_resumen.csv y clasificacion_sec.csv")

Mounted at /content/drive
Tickers sin CIK en la SEC (no se pueden consultar): ['FRBA', 'HIFS', 'NBN', 'RCBC', 'SSBI', 'TOWN']
Tickers a cubrir: 3282 | CIK únicos: 3237
Ya consultados: 0 | Pendientes: 3237 | Tiempo estimado: 22-43 min
200/3237 consultadas | 1.0 min
400/3237 consultadas | 2.0 min
600/3237 consultadas | 3.1 min
800/3237 consultadas | 4.2 min
1000/3237 consultadas | 5.2 min
1200/3237 consultadas | 6.2 min
1400/3237 consultadas | 7.2 min
1600/3237 consultadas | 8.2 min
1800/3237 consultadas | 9.2 min
2000/3237 consultadas | 10.3 min
2200/3237 consultadas | 11.4 min
2400/3237 consultadas | 12.6 min
2600/3237 consultadas | 13.7 min
2800/3237 consultadas | 14.8 min
3000/3237 consultadas | 15.9 min
3200/3237 consultadas | 17.0 min
3237/3237 consultadas | 17.2 min

=== RESULTADO DE LAS CONSULTAS ===
Empresas consultadas: 3237 | con error: 0
Empty DataFrame
Columns: [cik, error]
Index: []

Régimen según el último informe anual:
regimen
nacional             2819
extranjero        